In [ ]:
from typing import List

from vascx.segment import Segment, merge_segments
from vascx.retina import Retina
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from PIL import Image
from scipy.ndimage import distance_transform_edt
from vascx.analysis.network import get_segments

In [ ]:
r = Retina.from_file('../samples/av_hrf/01_h_new.png')
# r.load_fundus_image('./samples/av_hrf/images/01_h.jpg')
# r.load_disc()

In [ ]:
segments = r.veins.segments

In [ ]:
segments[10].id

In [ ]:
segments[10].neighbors

In [ ]:
segments[10].id

In [ ]:
# r.veins.plot_segments(text='median_diameter', color='median_diameter', show_id=False)

In [ ]:
def reset_visited(segments):
    for seg in segments:
        seg.visited = False
        
def recursive_set_diam(seg, ignore=None):
    if ignore is None:
        ignore = set()

    seg.visited = True
    if len(seg.neighbors) == 0:
        seg._thickest_diam = seg.mean_diameter
        return seg.mean_diameter
    else:
        neighbors = set(seg.neighbors) - ignore
        diam = max([seg.mean_diameter] + [recursive_set_diam(n, ignore=set(seg.neighbors)) for n in neighbors if not n.visited])
        seg._thickest_diam = diam
        return diam



def set_rank_1(seg):
    seg.rank = 1
    for s in seg.neighbors:
        if s.rank != 1:
            set_rank_1(s)

def recursive_set_rank(seg, value=0, ignore=None):
    if seg.rank is not None:
        return
    if ignore is None:
        ignore = set()
    
    seg.visited = True
    seg.rank = value

    neighbors = set([n for n in seg.neighbors if not n.visited]) - ignore
    if len(neighbors) == 0:
        # include parent's neighbors if there is no other choice
        neighbors = set([n for n in seg.neighbors if not n.visited])

    if len(neighbors) > 0:
        neighbors = sorted(neighbors, key=lambda x: x._agg_mean_diameter, reverse=True)
        recursive_set_rank(neighbors[0], value, ignore=set(seg.neighbors))

        for n in neighbors[1:]:
            recursive_set_rank(n, value+1, ignore=set(seg.neighbors))

def recursive_set_mean_width(seg, ignore=None):
    if ignore is None:
        ignore = set()

    seg.visited = True
    neighbors = set(seg.neighbors) - ignore
    neighbors = [n for n in neighbors if not n.visited]
    
    if len(neighbors) == 0:
        seg._agg_mean_diameter = seg.median_diameter
        seg._agg_length = seg.length
        return seg.median_diameter, seg.length
    else:
        
        tmp = [recursive_set_mean_width(n, ignore=set(seg.neighbors)) for n in neighbors]

        tmp = sorted(tmp, key=lambda x: x[0], reverse=True) # sort by mean diam
        max_agg_diameter, agg_length = tmp[0]
        seg._agg_mean_diameter = (max_agg_diameter * agg_length + seg.median_diameter * seg.length) / (agg_length + seg.length)
        seg._agg_length = agg_length + seg.length
        return seg._agg_mean_diameter, seg._agg_length
    


In [ ]:
r.veins.plot_segments(show_id=True)

In [ ]:
source_segments = [r.veins.segments[121], r.veins.segments[150], r.veins.segments[142]]
reset_visited(segments)
for seg in source_segments:
    recursive_set_mean_width(seg)
    # set_rank_1(seg)
reset_visited(segments)
for seg in source_segments:
    recursive_set_rank(seg)

In [ ]:
r.veins.plot_segments(color=lambda s: s.rank, text='_agg_length', show_id=False)

In [ ]:
r.veins.plot_segments(color=lambda s: s.rank, text='median_diameter', show_id=False)

In [ ]:
r.veins.plot_segments(color=lambda s: s.rank, text='_agg_mean_diameter', show_id=False)

In [ ]:
def recursive_merge(seg, ignore=None) -> (List[Segment], List[Segment]):
    if seg.visited == True:
        return [], []
    if ignore is None:
        ignore = set()
    
    seg.visited = True

    neighbors = set([n for n in seg.neighbors if not n.visited]) - ignore
    if len(neighbors) == 0:
        # include parent's neighbors if there is no other choice
        neighbors = set([n for n in seg.neighbors if not n.visited])

    open_segment, completed_segments = [seg], []
    if len(neighbors) > 0:
        neighbors = sorted(neighbors, key=lambda x: x._agg_mean_diameter, reverse=True)
        os, cs = recursive_merge(neighbors[0], ignore=set(seg.neighbors))

        open_segment += os
        completed_segments += cs

        for n in neighbors[1:]:
            os, cs = recursive_merge(n, ignore=set(seg.neighbors))

            completed_segments.append(merge_segments(*os))
            completed_segments += cs
    return open_segment, completed_segments

In [ ]:
source_segments = [r.veins.segments[153]]
reset_visited(segments)
for seg in source_segments:
    recursive_set_mean_width(seg)
reset_visited(segments)
for seg in source_segments:
    open_segment, completed_segments = recursive_merge(seg)

In [ ]:
completed_segments.append(merge_segments(*open_segment))

In [ ]:
r.veins._segments = completed_segments

In [ ]:
r.veins.plot_segments()

In [ ]:
recursive_merge(r.veins.segments[153])

In [ ]:
r.veins.plot_segments()

In [ ]:
r.veins.get_segment(2).neighbors

In [ ]:
r.veins.get_segment(157).neighbors

In [ ]:
# r.veins.plot_some_segments([157])

In [ ]:
r.veins.plot_segments(color_fn=lambda s: s.rank)

In [ ]:
r.veins.plot_segments(color_fn=lambda s: s.id % 20, cmap='tab20')

In [ ]:
r.veins.plot_segments(rank=1)

In [ ]:
r.veins.plot_segments()

In [ ]:
Image.fromarray(r.veins.skeleton).save('output.png')

In [ ]:
from vascx.analysis.network import make_adjacency_graph_8, connect_segment
from vascx.segment import Segment

In [ ]:
def plot_skl(skeleton, segments):
    arr = np.repeat((skeleton*255).astype(np.uint8)[:, :, None], 3, axis=2)

    for segment in segments:
        for pix in segment.pixels:
            arr[pix[0],pix[1],:] = [255,0,0]
        for pix in segment.ends:
            arr[pix[0],pix[1],:] = [0,255,0]
        for pix in segment.connectors:
            arr[pix[0],pix[1],:] = [0,0,255]


    # print(np.unique(arr))
    im = Image.fromarray(arr)
    # im = skeleton
    
    
    # # im = np.ma.masked_where(im == 0, im)


    # # ax.imshow(np.zeros_like(im))
    # ax.imshow(im, cmap='viridis')

    # return fig, ax
    return im

In [ ]:

def explore(graph, vertex, visited):

    pixels = set()
    ends = set()
    connectors = set()
    queue = [vertex]
    while queue:
        current = queue.pop()
        visited.add(current)

        pixels.add(current)
        neighbors = graph[current]

        if len(neighbors) > 2:
            ends.add(current)

            connectors.add(current)
            connectors.update(neighbors)
            continue

        has_neighbor = False
        for n in neighbors:
            if n not in visited:
                has_neighbor = True
                queue.append(n)

        if not has_neighbor:
            ends.add(current)

            connectors.add(current)
            connectors.update(neighbors)
    return pixels, ends, connectors

In [ ]:
def explore_n(n):
    pixels = set(zip(*np.where(r.veins.skeleton)))
    graph = make_adjacency_graph_8(pixels)

    segments = []
    visited = set()

    for vertex in list(graph.keys())[:n]:
        if vertex not in visited:
            
            pixels, ends, connectors = explore(graph, vertex, visited)
            segments.append(Segment(connect_segment(pixels, ends), frozenset(
                ends), frozenset(connectors - pixels)))

    return plot_skl(r.veins.skeleton, segments)

In [ ]:
explore_n(2).save('output.png')